# BIXI Station Map
Interactive map of all BIXI bike-share stations, plotted from `station_information.json`.

In [2]:
import json
import folium
import osmnx as ox
from shapely.geometry import Point

DATA_PATH = "/Volumes/Extreme SSD/SUMO Data/station_information.json"

In [3]:
with open(DATA_PATH) as f:
    raw = json.load(f)

stations = [
    s for s in raw["data"]["stations"]
    if s["lat"] != 0 and s["lon"] != 0
]

print(f"{len(stations)} valid stations loaded")

1059 valid stations loaded


In [4]:
lats = [s["lat"] for s in stations]
lons = [s["lon"] for s in stations]

bounds = [[min(lats), min(lons)], [max(lats), max(lons)]]

m = folium.Map(tiles="CartoDB positron")
m.fit_bounds(bounds, padding=(24, 24))

for s in stations:
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        radius=4,
        color="#e63e2a",
        fill=True,
        fill_color="#e63e2a",
        fill_opacity=0.85,
        weight=1.5,
        tooltip=folium.Tooltip(
            f"<b>{s['name']}</b><br>ID: {s['station_id']}<br>Capacity: {s['capacity']}",
            sticky=False,
        ),
    ).add_to(m)

m

## Rive-Sud (South Shore) Stations
Stations located south of the Saint-Laurent river — i.e., off the Island of Montréal and south of the island centroid.

In [5]:
# Fetch Island of Montréal boundary from OpenStreetMap
island_gdf = ox.geocode_to_gdf("Île de Montréal, Quebec, Canada")
island_poly = island_gdf.geometry.iloc[0]
island_centroid_lat = island_poly.centroid.y

# Classify each station
groups = {"on_island": [], "rive_sud": [], "laval_north": [], "sherbrooke": []}

for s in stations:
    pt = Point(s["lon"], s["lat"])
    if island_poly.contains(pt):
        groups["on_island"].append(s)
    elif s["lon"] > -73.0:          # far east cluster = Sherbrooke
        groups["sherbrooke"].append(s)
    elif s["lat"] < island_centroid_lat:
        groups["rive_sud"].append(s)
    else:
        groups["laval_north"].append(s)

for label, lst in groups.items():
    print(f"{label:15}: {len(lst):4} stations")

rive_sud = groups["rive_sud"]

on_island      :  846 stations
rive_sud       :   48 stations
laval_north    :  140 stations
sherbrooke     :   25 stations


In [6]:
# Map — Rive-Sud stations highlighted in blue, island stations in grey
all_lats = [s["lat"] for s in stations]
all_lons = [s["lon"] for s in stations]
bounds = [[min(all_lats), min(all_lons)], [max(all_lats), max(all_lons)]]

m2 = folium.Map(tiles="CartoDB positron")

rive_sud_ids = {s["station_id"] for s in rive_sud}

for s in stations:
    is_rs = s["station_id"] in rive_sud_ids
    folium.CircleMarker(
        location=[s["lat"], s["lon"]],
        radius=5 if is_rs else 3,
        color="#0057b8" if is_rs else "#aaaaaa",
        fill=True,
        fill_color="#0057b8" if is_rs else "#cccccc",
        fill_opacity=0.9 if is_rs else 0.5,
        weight=1.5,
        tooltip=folium.Tooltip(
            f"<b>{s['name']}</b><br>ID: {s['station_id']}<br>Capacity: {s['capacity']}"
            + (" <i>(Rive-Sud)</i>" if is_rs else ""),
            sticky=False,
        ),
    ).add_to(m2)

m2.fit_bounds(bounds, padding=(24, 24))
m2